In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import GridWorld, PolicyIteration, ValueIteration, GridWorldVisualizer, ExperimentRunner

In [2]:
# DONE: Create a custom GridWorld environment (4x4) using Gymnasium's API
# Configurable rewards, obstacles, and transition dynamics

env = GridWorld(
    size=4,
    obstacles=[(1, 1)],
    rewards={(3, 3): 1.0},
    default_reward=-0.04,
    terminal_states=[(3, 3)],
    stochastic=False,
)

print(f"Grid size: {env.size}x{env.size}")
print(f"States: {env.n_states}, Actions: {env.n_actions}")
print(f"Obstacles: {env.obstacles}")
print(f"Terminal states: {env.terminal_states}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

# Verify the transition model is built
P = env.get_transition_model()
print(f"\nTransition model entries: {sum(len(P[s]) for s in P)} (s,a) pairs")

# Quick sanity check: from (0,0) moving RIGHT
s = env.rc_to_state(0, 0)
print(f"\nP[state=0, action=RIGHT] = {P[s][env.RIGHT]}")

Grid size: 4x4
States: 16, Actions: 4
Obstacles: {(1, 1)}
Terminal states: {(3, 3)}
Observation space: Discrete(16)
Action space: Discrete(4)

Transition model entries: 64 (s,a) pairs

P[state=0, action=RIGHT] = [(1.0, 1, -0.04, False)]


In [3]:
# DONE: Make rewards, obstacles, and transition dynamics configurable
env_custom = GridWorld(
    size=5,
    obstacles=[(1, 1), (2, 2)],
    rewards={(4, 4): 10.0, (0, 4): -5.0},
    default_reward=-0.1,
    terminal_states=[(4, 4)],
    stochastic=False,
)

assert env_custom.size == 5
assert env_custom.obstacles == {(1, 1), (2, 2)}
assert env_custom.reward_map[(4, 4)] == 10.0
assert env_custom.reward_map[(0, 4)] == -5.0
assert env_custom.default_reward == -0.1
print("Configurable rewards, obstacles, and dynamics: OK")

Configurable rewards, obstacles, and dynamics: OK


In [4]:
# DONE: Include support for stochastic transitions
env_stoch = GridWorld(
    size=4, obstacles=[], rewards={(3, 3): 1.0}, default_reward=-0.04,
    terminal_states=[(3, 3)], stochastic=True, intended_prob=0.8,
)

P = env_stoch.get_transition_model()
s = env_stoch.rc_to_state(1, 1)  # interior cell, no boundary effects
transitions = P[s][env_stoch.RIGHT]

# Should have multiple outcomes (intended + perpendicular)
assert len(transitions) > 1, "Stochastic env should have multiple transition outcomes"

# Probabilities must sum to 1
total_prob = sum(t[0] for t in transitions)
assert abs(total_prob - 1.0) < 1e-9, f"Probabilities sum to {total_prob}, expected 1.0"

for prob, s_prime, reward, done in transitions:
    r, c = env_stoch.state_to_rc(s_prime)
    print(f"  P(s'=({r},{c}) | s=(1,1), a=RIGHT) = {prob}")
print("Stochastic transitions: OK")

  P(s'=(1,2) | s=(1,1), a=RIGHT) = 0.8
  P(s'=(0,1) | s=(1,1), a=RIGHT) = 0.09999999999999998
  P(s'=(2,1) | s=(1,1), a=RIGHT) = 0.09999999999999998
Stochastic transitions: OK


In [5]:
# DONE: Implement a method to extract or define the full transition model P(s',r|s,a)
P = env.get_transition_model()

assert len(P) == env.n_states, "P should have an entry for every state"
for s in range(env.n_states):
    assert len(P[s]) == env.n_actions, f"State {s} missing actions"
    for a in range(env.n_actions):
        probs = sum(t[0] for t in P[s][a])
        assert abs(probs - 1.0) < 1e-9, f"P[{s}][{a}] probs sum to {probs}"

print(f"Transition model: {env.n_states} states x {env.n_actions} actions, all probabilities valid")
print("Full transition model P(s',r|s,a): OK")

Transition model: 16 states x 4 actions, all probabilities valid
Full transition model P(s',r|s,a): OK


In [6]:
# DONE: Implement policy iteration (synchronous version)
pi = PolicyIteration(env, gamma=0.99, theta=1e-8)
V_pi, policy_pi = pi.solve(mode="sync")

print(f"Converged in {len(pi.value_history)} policy iteration rounds")
print(f"Eval sweeps per round: {pi.eval_iterations}")
print(f"Wall-clock times: {[f'{t:.4f}s' for t in pi.wall_clock_times]}")

# Display the value function as a grid
V_grid = V_pi.reshape(env.size, env.size)
print(f"\nValue function:\n{np.round(V_grid, 3)}")

# Display the policy as action names
action_names = ["UP", "RIGHT", "DOWN", "LEFT"]
policy_grid = np.array([action_names[a] for a in policy_pi]).reshape(env.size, env.size)
print(f"\nPolicy:\n{policy_grid}")

Converged in 7 policy iteration rounds
Eval sweeps per round: [1514, 2, 2, 2, 2, 2, 2]
Wall-clock times: ['0.0125s', '0.0001s', '0.0001s', '0.0001s', '0.0001s', '0.0001s', '0.0001s']

Value function:
[[0.755 0.803 0.851 0.9  ]
 [0.803 0.851 0.9   0.95 ]
 [0.851 0.9   0.95  1.   ]
 [0.9   0.95  1.    0.   ]]

Policy:
[['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['DOWN' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'UP']]


In [7]:
# DONE: Implement policy iteration (in-place version)
pi_inp = PolicyIteration(env, gamma=0.99, theta=1e-8)
V_inp, policy_inp = pi_inp.solve(mode="inplace")

print(f"Converged in {len(pi_inp.value_history)} policy iteration rounds")
print(f"Eval sweeps per round: {pi_inp.eval_iterations}")
print(f"Wall-clock times: {[f'{t:.4f}s' for t in pi_inp.wall_clock_times]}")

# Display the value function as a grid
V_grid_inp = V_inp.reshape(env.size, env.size)
print(f"\nValue function:\n{np.round(V_grid_inp, 3)}")

# Display the policy as action names
action_names = ["UP", "RIGHT", "DOWN", "LEFT"]
policy_grid_inp = np.array([action_names[a] for a in policy_inp]).reshape(env.size, env.size)
print(f"\nPolicy:\n{policy_grid_inp}")

# Verify sync and in-place agree
np.testing.assert_allclose(V_pi, V_inp, atol=1e-6)
np.testing.assert_array_equal(policy_pi, policy_inp)
print("\nSync and in-place values and policies match: OK")

Converged in 7 policy iteration rounds
Eval sweeps per round: [1514, 2, 2, 3, 2, 2, 1]
Wall-clock times: ['0.0099s', '0.0001s', '0.0001s', '0.0001s', '0.0001s', '0.0001s', '0.0001s']

Value function:
[[0.755 0.803 0.851 0.9  ]
 [0.803 0.851 0.9   0.95 ]
 [0.851 0.9   0.95  1.   ]
 [0.9   0.95  1.    0.   ]]

Policy:
[['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['DOWN' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'UP']]

Sync and in-place values and policies match: OK


In [8]:
# DONE: Implement value iteration (synchronous version)
vi = ValueIteration(env, gamma=0.99, theta=1e-8)
V_vi, policy_vi = vi.solve(mode="sync")

print(f"Converged in {vi.iterations} iterations")
print(f"Wall-clock times: {[f'{t:.4f}s' for t in vi.wall_clock_times[:5]]}{'...' if len(vi.wall_clock_times) > 5 else ''}")

# Display the value function as a grid
V_grid_vi = V_vi.reshape(env.size, env.size)
print(f"\nValue function:\n{np.round(V_grid_vi, 3)}")

# Display the policy as action names
action_names = ["UP", "RIGHT", "DOWN", "LEFT"]
policy_grid_vi = np.array([action_names[a] for a in policy_vi]).reshape(env.size, env.size)
print(f"\nPolicy:\n{policy_grid_vi}")

# Verify VI and PI agree
np.testing.assert_allclose(V_pi, V_vi, atol=1e-6)
np.testing.assert_array_equal(policy_pi, policy_vi)
print("\nVI sync and PI sync values and policies match: OK")

Converged in 7 iterations
Wall-clock times: ['0.0003s', '0.0002s', '0.0001s', '0.0000s', '0.0000s']...

Value function:
[[0.755 0.803 0.851 0.9  ]
 [0.803 0.851 0.9   0.95 ]
 [0.851 0.9   0.95  1.   ]
 [0.9   0.95  1.    0.   ]]

Policy:
[['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['DOWN' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'UP']]

VI sync and PI sync values and policies match: OK


In [9]:
# DONE: Implement value iteration (in-place version)
vi_inp = ValueIteration(env, gamma=0.99, theta=1e-8)
V_vi_inp, policy_vi_inp = vi_inp.solve(mode="inplace")

print(f"Converged in {vi_inp.iterations} iterations")
print(f"Wall-clock times: {[f'{t:.4f}s' for t in vi_inp.wall_clock_times[:5]]}{'...' if len(vi_inp.wall_clock_times) > 5 else ''}")

# Display the value function as a grid
V_grid_vi_inp = V_vi_inp.reshape(env.size, env.size)
print(f"\nValue function:\n{np.round(V_grid_vi_inp, 3)}")

# Display the policy as action names
action_names = ["UP", "RIGHT", "DOWN", "LEFT"]
policy_grid_vi_inp = np.array([action_names[a] for a in policy_vi_inp]).reshape(env.size, env.size)
print(f"\nPolicy:\n{policy_grid_vi_inp}")

# Verify in-place and sync agree
np.testing.assert_allclose(V_vi, V_vi_inp, atol=1e-6)
np.testing.assert_array_equal(policy_vi, policy_vi_inp)
print("\nVI sync and VI in-place values and policies match: OK")

Converged in 7 iterations
Wall-clock times: ['0.0001s', '0.0000s', '0.0000s', '0.0000s', '0.0000s']...

Value function:
[[0.755 0.803 0.851 0.9  ]
 [0.803 0.851 0.9   0.95 ]
 [0.851 0.9   0.95  1.   ]
 [0.9   0.95  1.    0.   ]]

Policy:
[['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['DOWN' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'DOWN']
 ['RIGHT' 'RIGHT' 'RIGHT' 'UP']]

VI sync and VI in-place values and policies match: OK


In [18]:
# DONE: Use NumPy arrays for value functions V(s) and policies π(s)
# Both PolicyIteration and ValueIteration store V and policy as np.ndarray internally.
# Verify the types and shapes returned by each solver.

pi = PolicyIteration(env, gamma=0.99, theta=1e-8)
V_pi, policy_pi = pi.solve(mode="sync")

vi = ValueIteration(env, gamma=0.99, theta=1e-8)
V_vi, policy_vi = vi.solve(mode="sync")

for label, V, policy in [("PI", V_pi, policy_pi), ("VI", V_vi, policy_vi)]:
    assert isinstance(V, np.ndarray), f"{label} V is not ndarray"
    assert isinstance(policy, np.ndarray), f"{label} policy is not ndarray"
    assert V.shape == (env.n_states,), f"{label} V shape mismatch"
    assert policy.shape == (env.n_states,), f"{label} policy shape mismatch"
    assert V.dtype == np.float64, f"{label} V dtype mismatch"
    assert policy.dtype == np.int64 or policy.dtype == np.int_, f"{label} policy dtype mismatch"
    print(f"{label}: V {V.dtype}{V.shape}, policy {policy.dtype}{policy.shape}")

# Value history entries are also NumPy arrays
for arr in pi.value_history:
    assert isinstance(arr, np.ndarray)
for arr in vi.value_history:
    assert isinstance(arr, np.ndarray)

print("All value functions and policies are NumPy arrays: OK")

PI: V float64(16,), policy int64(16,)
VI: V float64(16,), policy int64(16,)
All value functions and policies are NumPy arrays: OK


In [11]:
# TODO: Visualize the value function at each iteration as a heatmap (Matplotlib)


In [12]:
# TODO: Visualize the policy at each iteration using arrows (quiver plots)


In [13]:
# TODO: Visualize convergence curves comparing iteration count and wall-clock time


In [14]:
# TODO: Test on a deterministic GridWorld configuration


In [15]:
# TODO: Test on a stochastic GridWorld configuration (e.g., 80% intended, 10% perpendicular)


In [16]:
# TODO: Apply DP implementations to Gymnasium's FrozenLake-v1 using its transition dynamics


In [17]:
# TODO: Document which algorithm converges faster under different conditions and explain why